# Deutsch-Jozsa Algorithm

Given a function $f:\{0,1\}^n \to \{0,1\}$ promised to be either **constant** (same output for all inputs) or **balanced** (outputs 0 for exactly half the inputs and 1 for the other half), determine which it is.

Classically this requires up to $2^{n-1}+1$ evaluations of $f$ in the worst case. The worst case arises when the function is actually balanced, but you happen to keep picking input values that all map to the same output — say, all 0. After $2^{n-1}$ queries all returning 0 you still cannot be certain: the function might be constant-0, or it might be balanced and you have just sampled the half that returns 0. Only when a further query also returns 0 can you be sure. Since the problem guarantees the function is either constant or balanced — nothing in between — seeing $2^{n-1}+1$ inputs all returning the same value proves it must be constant, because a balanced function has exactly half its inputs mapping to each output and no more.

The Deutsch-Jozsa algorithm answers the question with a single evaluation.

Throughout this notebook we use **4 input qubits** (`a`, `b`, `c`, `d`) and one **ancilla qubit** (`e`). The ancilla is a helper qubit used to hold the function's output: the oracle writes $f(x)$ into it via $U_f\,|x\rangle|y\rangle = |x\rangle|y \oplus f(x)\rangle$. The ancilla is initialised to $|1\rangle$ and then put into the $|{-}\rangle$ state (the $-1$ eigenstate of $\sigma_x$) by applying H. In practice qubits are typically prepared as $|0\rangle$, so this is equivalent to initialising to $|0\rangle$ and flipping with an X gate before the H — the circuits below show the $|1\rangle$ initialisation directly to keep the diagrams uncluttered. At the end of the algorithm we do not measure the ancilla — we are only interested in whether the input qubits have changed, so the measurement is performed on `a`, `b`, `c`, and `d` only.

## Constant functions: nothing to see

For a constant function the oracle either does nothing ($f=0$) or unconditionally flips the ancilla ($f=1$). Either way the four input qubits `a`–`d` are completely untouched by the oracle.

**Constant $f = 0$** — the oracle is the identity, and the ancilla `e` passes through untouched:

<img src="images/deutsch-jozsa-constant-0.svg" alt="Deutsch-Jozsa circuit for constant f=0: qubits a-d start in |0⟩, ancilla e in |1⟩; H on every qubit, an empty oracle, H on every qubit again; a-d are measured, e is not" width="420"/>

**Constant $f = 1$** — the oracle unconditionally flips the ancilla; the input qubits are still untouched:

<img src="images/deutsch-jozsa-constant-1.svg" alt="Deutsch-Jozsa circuit for constant f=1: same as f=0 but the oracle applies X to the ancilla only; a-d are measured, e is not" width="420"/>

The final H on `e` is not strictly required — we discard the ancilla's value and do not measure it. It is included to make the algorithm's structure explicit: every qubit enters the oracle in the Hadamard basis and exits through a matching H. This symmetry is what the balanced case exploits, and it is clearer when the ancilla is shown to participate in it equally.

In both cases the input qubits experience nothing but H·H = I — the Hadamard is its own inverse. They were initialised to $|0\rangle$ and undergo no net transformation, so they all measure as $|0\rangle$. There is nothing surprising here.

## Balanced functions: phase kickback as a magic trick

For the balanced case, the oracle contains CNOT gates from the input qubits to the ancilla. The circuit looks like:

<img src="images/deutsch-jozsa-balanced.svg" alt="Deutsch-Jozsa circuit for balanced f = a XOR b: H on every qubit, then CNOTs from a and from b targeting the ancilla e, then H on every qubit; a-d are measured, e is not" width="480"/>

Here we use four input qubits and a balanced function $f(a,b,c,d) = a \oplus b$ — the output depends on only the first two inputs. Qubits `c` and `d` pass through the oracle unconnected.

**Why XOR?** You might expect the oracle to simply set `e` to $f(x)$, but quantum gates must be unitary — they are reversible transformations, not assignments. Setting a qubit's value outright would require a measurement, which means the environment interacts with the qubit and causes decoherence, destroying the superposition the algorithm depends on. Instead, the oracle adds $f(x)$ to the ancilla's current value modulo 2: $|y\rangle \mapsto |y \oplus f(x)\rangle$. This is reversible (apply it twice and you get back to $|y\rangle$), so it can be implemented with unitary gates. The ancilla's initial value is preserved in the sense that the transformation is always undoable — the whole circuit can be run backwards.

At first glance it is not obvious why measuring the input qubits would reveal anything. The input qubits are the **controls** in the oracle CNOTs — in notebook 007 we established that the control qubit's Z observable is left unchanged by CNOT. How can applying two Hadamards around an oracle that doesn't touch the input qubits' Z observables produce a non-trivial measurement result?

The answer is that **the Hadamard gates swap the X and Z observables**. In notebook 007 we also saw that CNOT *does* affect the control qubit's X observable — it becomes entangled with the target's X observable. The first Hadamard turns the input qubits' Z into X, so the oracle's CNOT affects what will become the Z observable; the second Hadamard turns it back into Z for the measurement. The whole H–CNOT–H sandwich is equivalent to running the CNOT with the control and target reversed. The magic trick is that the entire oracle section — CNOT gates and all — runs in the Hadamard basis. Every qubit is in the Hadamard basis while the oracle runs, and the matching H on the way out undoes the basis change. Showing the final H on the ancilla makes this uniform: the algorithm is just ordinary CNOTs sandwiched between Hadamards on every wire.

Redrawn without the Hadamards and with the CNOT arrows flipped:

<img src="images/deutsch-jozsa-balanced-flipped.svg" alt="Equivalent circuit with no Hadamards: the ancilla e in |1⟩ is the control of CNOTs targeting a and b; a-d are measured, e is not" width="300"/>

In this equivalent picture the ancilla is the control qubit in state $|1\rangle$, and the input qubits are the targets. A CNOT with control in $|1\rangle$ always flips its target. Qubits `a` and `b` are connected to the ancilla and get flipped to $|1\rangle$; qubits `c` and `d` are not connected and stay at $|0\rangle$. The measurement result $[1, 1, 0, 0]$ directly identifies which inputs the function depends on.

It is not at all surprising that this produces a non-trivial result — the ancilla in $|1\rangle$ just flips the relevant qubits. The Hadamard gates are the sleight of hand that hides this mechanism inside what looks like a forward evaluation of $f$.

In [6]:
from qubit import Qubit
from gates_single import hadamard, qnot
from gates_multi import cnot
from IPython.display import display

qa = Qubit.qubit_time_0('a')
qb = Qubit.qubit_time_0('b')
qc = Qubit.qubit_time_0('c')
qd = Qubit.qubit_time_0('d')
qe = Qubit.qubit_time_0('e')  # ancilla

qe = qnot(qe)                  # |0⟩ → |1⟩

qa, qb, qc, qd, qe = [hadamard(q) for q in [qa, qb, qc, qd, qe]]

# Oracle: f(a,b,c,d) = a ⊕ b — only a and b are connected to the ancilla
qa, qe = cnot(qa, qe)
qb, qe = cnot(qb, qe)

qa, qb, qc, qd = [hadamard(q) for q in [qa, qb, qc, qd]]

display(qa)
display(qb)
display(qc)
display(qd)

Qubit(a, a1, -a2*e3, -a3*e3)

Qubit(b, b1, -b2*e3, -b3*e3)

Qubit(c, c1, c2, c3)

Qubit(d, d1, d2, d3)

## Reading the result

The Z observables of the input qubits after the circuit are:

- `a.Z` = $-\sigma_z^{(a)}\,\sigma_z^{(e)}$ — expectation $-1$, measures **1**
- `b.Z` = $-\sigma_z^{(b)}\,\sigma_z^{(e)}$ — expectation $-1$, measures **1**
- `c.Z` = $\sigma_z^{(c)}$ — expectation $+1$, measures **0**
- `d.Z` = $\sigma_z^{(d)}$ — expectation $+1$, measures **0**

The $-\sigma_z^{(e)}$ factor in `a` and `b` is the observable fingerprint of the ancilla's $|{-}\rangle$ preparation. When $qnot$ was applied to the ancilla, its Z observable became $-\sigma_z^{(e)}$. After the Hadamard this moved into the X position, and then the oracle's CNOT gates copied it into the X observables of `a` and `b`. The final Hadamard on the input qubits moved it into the Z position, where the measurement reads it out as $-1$.

Qubits `c` and `d` have no connection to the ancilla in the oracle, so the $-\sigma_z^{(e)}$ factor never reaches them. Their Z observables remain $\sigma_z^{(c)}$ and $\sigma_z^{(d)}$, measuring as $+1$ (i.e. $|0\rangle$).

The measurement result $[1, 1, 0, 0]$ is non-zero, so the function is **balanced** — and the pattern of which qubits measured 1 directly identifies the two inputs the function depends on.